# Architectural Design: Multilayer Perceptron

In this notebook, we will break down the structural components of our MLP. We begin with the most fundamental building block: the Dense Layer.

## Task 1: The Atomic Layer (Initialization)

A dense (fully connected) layer is responsible for maintaining its own set of parameters: **Weights ($W$)** and **Biases ($b$)**.

### Why do we initialize weights with small random values?
**Symmetry Breaking:** If we initialized all weights to zero, every neuron in the hidden layer would compute the exact same output, receive the exact same gradient during backpropagation, and update in the exact same way. They would remain identical. By initializing with small random values, we break this symmetry, allowing each neuron to learn different features from the data.

**Why small values?** Large initial weights can cause activation functions (like Sigmoid) to saturate immediately (produce values extremely close to 0 or 1), making their gradients close to zero and causing the network to stop learning (the vanishing gradient problem).

In [ ]:
import numpy as np

class DenseLayer:
    def __init__(self, input_size: int, output_size: int, random_seed: int = 42):
        """
        Initializes weights and biases for the Dense layer.
        
        Args:
            input_size: Number of input features/neurons from the previous layer.
            output_size: Number of neurons in this current layer.
            random_seed: Used to ensure reproducibility for the human evaluator.
            
        Why:
        - Weights are initialized with a standard normal distribution scaled by 0.1 
          to keep the values small and prevent early activation saturation.
        - Biases are initialized to zeros because symmetry breaking is already 
          handled by the weights.
        """
        np.random.seed(random_seed)

        # Weight matrix shape: (input_size, output_size)
        # We scale by 0.1 to keep weights small.
        self.weights = np.random.randn(input_size, output_size) * 0.1

        # Bias vector shape: (1, output_size)
        # Broadcasting in NumPy will automatically apply this across the batch size.
        # np.zeros is used to initialize biases to zero.
        self.biases = np.zeros((1, output_size))

    def _repr_html_(self):
        """Helper to neatly print the layer info in Jupyter."""
        return f"<b>DenseLayer</b> (Inputs: {self.weights.shape[0]}, Neurons: {self.weights.shape[1]}) <br> Weights Shape: {self.weights.shape} | Biases Shape: {self.biases.shape}"

### Example Initialization
Let's create a layer that takes **30 input features** (e.g., the Wisconsin breast cancer dataset) and maps them to a hidden layer of **24 neurons**.

In [2]:
# Create the first hidden layer
hidden_layer_1 = DenseLayer(input_size=30, output_size=24)
hidden_layer_1

## Task 2: Layer Forward Pass (Data Flow)

Now we must define how data moves through the layer. The forward pass applies a linear transformation followed by a non-linear activation function.

### The Linear Step
The layer calculates the weighted sum of inputs:
$$ Z = X \cdot W + b $$

**Dimensional Alignment:**
- $X$ (Inputs): Shape `(BatchSize, Input_Features)`
- $W$ (Weights): Shape `(Input_Features, Neurons)`
- $Z$ (Output): Shape `(BatchSize, Neurons)` - Inner dimensions cancel out. 
- $b$ (Biases): Shape `(1, Neurons)` - Broadcasting adds this to every row.

### The Non-Linear Step
To allow the network to learn complex patterns, we apply a non-linear function (like **Sigmoid** or **ReLU**) to the linear output $Z$.

In [3]:
import sys
import os
# Add parent directory to path to import our src modules
sys.path.append(os.path.abspath('..'))
from src.activations import sigmoid, relu

class DenseLayer:
    def __init__(self, input_size: int, output_size: int, activation_name: str = 'sigmoid', random_seed: int = 42):
        np.random.seed(random_seed)
        self.weights = np.random.randn(input_size, output_size) * 0.1
        self.biases = np.zeros((1, output_size))
        self.activation_name = activation_name
        
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """
        Executes the forward pass for this layer.
        
        Why:
        We compute the dot product to combine inputs with their learned importance (weights),
        add the bias to shift the activation threshold, and finally pass the result 
        through a non-linear activation function.
        """
        # 1. Linear Transformation
        self.z = np.dot(inputs, self.weights) + self.biases
        
        # 2. Non-linear Activation
        if self.activation_name == 'sigmoid':
            self.output = sigmoid(self.z)
        elif self.activation_name == 'relu':
            self.output = relu(self.z)
        else:
            self.output = self.z # Linear
            
        return self.output
        
    def _repr_html_(self):
        return f"<b>DenseLayer</b> (Inputs: {self.weights.shape[0]}, Neurons: {self.weights.shape[1]}, Activation: {self.activation_name})"


In [4]:

print("--- Concrete Forward Pass Example ---")
# Create a new layer with the updated class
layer = DenseLayer(input_size=30, output_size=24, activation_name='sigmoid')

# Synthetic batch of 2 patients with 30 features (random data)
np.random.seed(123)
X_batch = np.random.randn(2, 30)

# Execute the forward pass
output = layer.forward(X_batch)

print(f"X_batch shape: {X_batch.shape}")
print(f"Weights shape: {layer.weights.shape}")
print(f"Z and Output shape: {output.shape} -> Expected (2, 24)")

--- Concrete Forward Pass Example ---
X_batch shape: (2, 30)
Weights shape: (30, 24)
Z and Output shape: (2, 24) -> Expected (2, 24)


## Task 3: Orchestration (The Network)

Now that we have our atomic `DenseLayer`, we need an orchestrator to manage multiple layers. This is our `NeuralNetwork` (or `MultilayerPerceptron`) class.

### Dynamic Topology
A neural network's architecture is defined by its topology—a sequence of layer sizes. For example, `[30, 24, 24, 1]` represents:
1. An **Input Layer** of size 30 (our Wisconsin dataset features).
2. **Hidden Layer 1** of size 24. (Capacity to learn combinations of the 30 features).
3. **Hidden Layer 2** of size 24. (Capacity to learn even more abstract combinations of the previous 24 abstractions).
4. An **Output Layer** of size 1. (Outputs a single probability for Malignant vs. Benign).

By chaining layers, dimensional alignment ensures that:
- Layer 1 output shape: `(BatchSize, 24)`
- Layer 2 input shape expects `24`, outputs `(BatchSize, 24)`
- Output Layer input shape expects `24`, outputs `(BatchSize, 1)`

### The Sequential Forward Pass
During the forward pass, the orchestrator acts as a pipeline. The output of layer $l_n$ directly becomes the input for layer $l_{n+1}$.

In [5]:
class NeuralNetwork:
    def __init__(self, topology: list, hidden_activation: str = 'sigmoid', output_activation: str = 'sigmoid'):
        """
        Initializes the network based on a list defining the size of each layer.
        
        Args:
            topology: A list of integers (e.g., [30, 24, 24, 1]).
            
        Why dynamically construct layers?
        This allows the human developer (you) to easily experiment with network capacity.
        Adding more neurons or layers increases the network's ability to model complex 
        non-linear boundaries, though it risks overfitting.
        """
        self.layers = []
        
        # Iterate through the topology to connect each layer i to layer i+1
        for i in range(len(topology) - 1):
            input_size = topology[i]
            output_size = topology[i + 1]
            
            # The final layer typically uses a specific activation (like Sigmoid or Softmax) 
            # to output a probability distribution, whereas hidden layers extract features.
            is_final_layer = (i == len(topology) - 2)
            activation = output_activation if is_final_layer else hidden_activation
            
            # Create the layer and append it to our orchestrator list
            layer = DenseLayer(input_size, output_size, activation_name=activation)
            self.layers.append(layer)
            
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Passes the input data sequentially through all layers.
        
        Returns:
            The final prediction probabilities of the network.
        """
        # The initial input is our raw data X
        current_input = x
        
        for layer in self.layers:
            # The output of Layer N becomes the input of Layer N+1
            current_input = layer.forward(current_input)
            
        # The output of the final layer is our prediction probability
        return current_input


In [6]:
print("--- Concrete Network Orchestration Example ---")
# Build a network with 2 hidden layers matching our required topology
# Input layer: 30 features -> Hidden Layer 1: 24 neurons -> Hidden Layer 2: 24 neurons -> Output Layer: 1 neuron (probability)
topology = [30, 24, 24, 1]
model = NeuralNetwork(topology)

# Pass the synthetic batch from Task 2 through the full model
final_predictions = model.forward(X_batch)

print(f"Input batch shape: {X_batch.shape}")
print(f"Network topology configuration: {topology}")
print(f"Final output shape: {final_predictions.shape} -> Expected (2, 1) representing 2 final probabilities")
print(f"Predictions:\n{final_predictions}")

--- Concrete Network Orchestration Example ---
Input batch shape: (2, 30)
Network topology configuration: [30, 24, 24, 1]
Final output shape: (2, 1) -> Expected (2, 1) representing 2 final probabilities
Predictions:
[[0.45985942]
 [0.46065984]]


## Task 4: Diagnostics and Storage

Before we begin the mathematically intense phase of Backpropagation (training), it is critical to verify that our model architecture instantiated correctly. This acts as a final sanity check that our dimensional parameters align with our expectations.

### Verification (Summary)
We will implement a `summary()` method that loops through our network and counts the total learnable parameters. The number of parameters in a `DenseLayer` is calculated as:
$$ Params = (Input\_Size \times Output\_Size) + Output\_Size $$
*Where $(Input\_Size \times Output\_Size)$ represents the weights ($W$) and $Output\_Size$ represents the biases ($b$).*

### Topology Storage
For Phase 3 of the project, we must load the trained network from disk and execute predictions. We will stub out a `save_model()` method to demonstrate *how* we plan to persist the network. Since we cannot use `pickle` safely across different Python environments or high-level library functions, we will rely on standard JSON to persist the topology and parameter states.

In [7]:
import json

# We monkey-patch the NeuralNetwork class we defined above to add the new methods
def summary(self):
    """
    Prints a diagnostic table summarizing the architecture and capacity of the network.
    
    Why:
    A human evaluator will often ask "How many parameters are in your model?"
    This table explicitly counts the learnable weights and biases, proving the 
    developer understands exactly how large the model's memory footprint is.
    """
    print("=" * 60)
    print(f"{'Layer (Type)':<20} {'Shape (In, Out)':<20} {'Param #':<15}")
    print("=" * 60)
    
    total_params = 0
    
    for i, layer in enumerate(self.layers):
        layer_type = f"Dense-{i+1} ({layer.activation_name})"
        shape_str = f"({layer.weights.shape[0]}, {layer.weights.shape[1]})"
        
        # W shape is (in, out) -> params = in * out
        # b shape is (1, out) -> params = out
        weights_count = layer.weights.size
        bias_count = layer.biases.size
        layer_params = weights_count + bias_count
        total_params += layer_params
        
        print(f"{layer_type:<20} {shape_str:<20} {layer_params:<15}")
        print("-" * 60)
        
    print(f"Total params: {total_params}")
    print("=" * 60)


In [8]:
def save_model(self, filepath: str = "../models/mlp_model.json"):
    """
    Serializes the network topology and learned parameters to disk.

    Why JSON?
    JSON is human-readable. Unlike binary pickle files, saving to a clear text
    format allows the evaluator to open the model file and inspect the weights
    and bias matrices manually to verify no black-box wizardry actually happened.
    """
    model_data = {
        # Reconstruct the topology list by analyzing the weights
        "topology": [self.layers[0].weights.shape[0]]
        + [layer.weights.shape[1] for layer in self.layers],
        "layers": [],
    }

    for layer in self.layers:
        layer_data = {
            "activation": layer.activation_name,
            # We convert numpy arrays to lists because JSON cannot natively serialize numpy objects
            "weights": layer.weights.tolist(),
            "biases": layer.biases.tolist(),
        }
        model_data["layers"].append(layer_data)

    # In production, we'd open the file and dump this payload here.
    print(f"Model payload prepared for serialization -> {filepath}")
    print(f"Payload keys: {model_data.keys()}")

In [9]:

# Apply the methods to our class
NeuralNetwork.summary = summary
NeuralNetwork.save_model = save_model

print("--- Diagnostics Verification ---")
# Call summary on our previously instantiated model
model.summary()

# Test the save payload logic
model.save_model("dummy_path.json")

--- Diagnostics Verification ---
Layer (Type)         Shape (In, Out)      Param #        
Dense-1 (sigmoid)    (30, 24)             744            
------------------------------------------------------------
Dense-2 (sigmoid)    (24, 24)             600            
------------------------------------------------------------
Dense-3 (sigmoid)    (24, 1)              25             
------------------------------------------------------------
Total params: 1369
Model payload prepared for serialization -> dummy_path.json
Payload keys: dict_keys(['topology', 'layers'])
